In [ ]:
def make_tiled_stats(run_id:str,
                     rep_list:str,
                     base_data_directory:str = '/efs/scratch/methyl_qc_data/',
                     keep_chr_names:list = default_chr_list,
                     tile_win_size:int = 1000,
                     tile_step_size:int = 1000,
                     tile_cov_bases:int = 0,
                     verbose:bool = False) -> (pd.DataFrame, pd.DataFrame):

    import rpy2
    from rpy2.robjects.packages import importr
    import os
    import rpy2.rinterface as rinterface
    as_char = rinterface.baseenv['as.character']
 
    # Indices for (tiled) methylation R object
    chromosome_column_index = 0
    start_column_index = 1
    end_column_index = 2
    strand_column_index = 3
    coverage_column_index = 4
    meth_column_index = 5
    unmeth_column_index = 6
    
    methylKit = importr('methylKit')
    rutils = importr('utils')
    genomation = importr('genomation')
   
    methobj_fn = base_data_directory+run_id+'/methobj.rds'

    # We define an R function that is a reformulated version of getCorrelation from MethylKit
    # to clean up the output format
    rpy2.robjects.r('''
    getCorrelation_local <- function(object, method = c("pearson","kendall","spearman")){
        meth.mat = getData(object)[, object@numCs.index]/
            (getData(object)[,object@numCs.index] + 
             getData(object)[,object@numTs.index])                                      

        names(meth.mat)=object@sample.ids
        method <- match.arg(method)
        return_data = cor(meth.mat,method=method, use = "na.or.complete")
        return(return_data)
    }''')

    methobj = rpy2.robjects.r('methobj = unite(readRDS(\"'+methobj_fn+'\"),min.per.group=1L)')
    meth_correlations = rpy2.robjects.r('getCorrelation_local(methobj)')

    meth_correlations_df = pd.DataFrame(np.array(meth_correlations), index=rep_list, columns=rep_list)

    methobj_lowCov_fn = base_data_directory+run_id+'/methobj_lowcov.rds'
    methobj_tiles_fn = base_data_directory+run_id+'/methobj_lowcov_tiles.rds'

    if os.path.isfile(methobj_tiles_fn):
        methobj_tiles = rpy2.robjects.r.readRDS(methobj_tiles_fn)
    else:
        methobj_lowCov = rpy2.robjects.r.readRDS(methobj_lowCov_fn)
        methobj_tiles = methylKit.tileMethylCounts(methobj_lowCov, win_size = tile_win_size, step_size = tile_step_size, cov_bases = tile_cov_bases)
        rpy2.robjects.r.saveRDS(methobj_tiles,file=methobj_tiles_fn)

    tile_beta_df_fn = base_data_directory+run_id+'/tile_beta_df.csv'

    if os.path.isfile(tile_beta_df_fn):
        tile_beta_df = pd.read_csv(tile_beta_df_fn, index_col=0)
    else:
        if verbose:
            print('tile_beta_df.csv does not exist. Creating...')
        tile_beta_df = tile_beta_to_csv(rep_list, methobj_tiles, keep_chr_names, verbose, tile_beta_df_fn = tile_beta_df_fn)

    return (tile_beta_df, meth_correlations_df)

In [ ]:
def tile_beta_to_csv(rep_list:list,
                     methobj_tiles, #:rpy2.robjects.vectors.ListVector,
                     keep_chr_names:list = default_chr_list,
                     verbose:bool = False,
                     save_file:bool = True,
                     tile_beta_df_fn:str = 'tile_beta_df.csv') -> pd.DataFrame:
    '''
    Function taking rpy2 object methobj_tiles and saving the tiles beta values to a csv named tile_beta_df_fn.
    '''

    import numpy as np
    import pandas as pd
    from methylqc.utils import beta
    import rpy2
    from rpy2.robjects.packages import importr
    import rpy2.rinterface as rinterface
    as_char = rinterface.baseenv['as.character']
    rutils = importr('utils')

    # Indices for (tiled) methylation R object
    chromosome_column_index = 0
    start_column_index = 1
    end_column_index = 2
    strand_column_index = 3
    coverage_column_index = 4
    meth_column_index = 5
    unmeth_column_index = 6

    tile_betaL = []
    for i,tiled_counts_obj in enumerate(methobj_tiles):
        if verbose:
            print('Tiling beta values for '+rep_list[i])

        # We must be careful to index the R object with 1-based indexing, so the range of indices is shifted by 1.
        chr_list = [as_char(tiled_counts_obj[chromosome_column_index].rx[int(i)])[0] for i in np.arange(1, len(tiled_counts_obj[chromosome_column_index])+1)]

        coord_matrix = pd.DataFrame({
            'chr': chr_list,
            'start': list(map(str,np.array(tiled_counts_obj[start_column_index]))),
            'end': list(map(str,np.array(tiled_counts_obj[end_column_index]))),
            'strand': list(map(str,np.array(tiled_counts_obj[strand_column_index])))})

        is_in_keep_chr = coord_matrix.loc[:,'chr'].isin(keep_chr_names)
        meth_column = pd.Series(tiled_counts_obj[meth_column_index]).where(is_in_keep_chr).dropna().values
        unmeth_column = pd.Series(tiled_counts_obj[unmeth_column_index]).where(is_in_keep_chr).dropna().values
        coord_matrix = coord_matrix.where(is_in_keep_chr).dropna().values
        tile_index =  list(map(lambda x: '_'.join(x), coord_matrix))

        tile_beta_aux = pd.Series(beta(meth_column, unmeth_column, 0), index=tile_index, name = rep_list[i])

        tile_betaL.append(tile_beta_aux)

    dups = [q.index.where(q.index.duplicated()).dropna().values for q in tile_betaL]
    tile_betaL_filt = [tile_betaL[i].mask(tile_betaL[i].index.isin(q)).dropna() for i,q in enumerate(dups)]
    tile_beta_df = pd.concat(tile_betaL_filt, axis=1).fillna(0)

    if save_file:
        tile_beta_df.to_csv(tile_beta_df_fn)
        
    return tile_beta_df

# tileMethylCounts

In python from rpy2, this ends up being 

```python
methobj_lowCov = rpy2.robjects.r.readRDS(methobj_lowCov_fn)

methobj_tiles = methylKit.tileMethylCounts(methobj_lowCov, win_size = tile_win_size, step_size = tile_step_size, cov_bases = tile_cov_bases)
```

I think it would be best to just put this in a simple R script? Now just to figure out how to get that to work..

In [1]:
%load_ext rpy2.ipython

In [2]:
%%R 
library(methylKit)

R[write to console]: Loading required package: GenomicRanges

R[write to console]: Loading required package: stats4

R[write to console]: Loading required package: BiocGenerics

R[write to console]: 
Attaching package: ‘BiocGenerics’


R[write to console]: The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


R[write to console]: The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min


R[write to console]: Loading required package: S4Vectors

R[write to console]: 
Attaching package: ‘S4Vectors’


R[write to console]: The following objects are masked from ‘package:base’

In [3]:
%%R
methobj_lowCov <- readRDS("/efs/scratch/jeffrey/code/github.com/exosomedx/methylseq/tests/results/test_NSQCAM529_subsample_100k_200k/methylqc/methobj_lowcov.rds")

In [4]:
%%R 
library(tidyverse)

R[write to console]: ── Attaching packages ─────────────────────────────────────── tidyverse 1.3.1 ──

R[write to console]: ✔ ggplot2 3.4.2     ✔ purrr   1.0.1
✔ tibble  3.2.1     ✔ dplyr   1.1.2
✔ tidyr   1.3.0     ✔ stringr 1.5.0
✔ readr   2.1.4     ✔ forcats 1.0.0

R[write to console]: ── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::collapse()   masks IRanges::collapse()
✖ dplyr::combine()    masks BiocGenerics::combine()
✖ dplyr::desc()       masks IRanges::desc()
✖ tidyr::expand()     masks S4Vectors::expand()
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::first()      masks S4Vectors::first()
✖ dplyr::lag()        masks stats::lag()
✖ ggplot2::Position() masks BiocGenerics::Position(), base::Position()
✖ purrr::reduce()     masks GenomicRanges::reduce(), IRanges::reduce()
✖ dplyr::rename()     masks S4Vectors::rename()
✖ dplyr::select()     masks methylKit::select()
✖ dplyr::slice()      masks IRanges::slice()
✖ tidyr::unite()      

In [5]:
%%R 
methobj_tiles <- tileMethylCounts(
	methobj_lowCov,
	win_size = 1000, 
	step_size = 1000, 
	cov_bases = 0
) 

In [6]:
%%R 
methobj_tiles 

methylRawList object with 2 methylRaw objects

methylRaw object with 1700 rows
--------------
   chr   start     end strand coverage numCs numTs
1 chr1  923001  924000      *        3     2     1
2 chr1 1108001 1109000      *       23    23     0
3 chr1 1321001 1322000      *        3     3     0
4 chr1 1425001 1426000      *       10     5     5
5 chr1 1439001 1440000      *        9     6     3
6 chr1 1554001 1555000      *        6     6     0
--------------
sample.id: Healthy_1_200k 
assembly: hg38 
context: CpG 
resolution: region 

methylRaw object with 331 rows
--------------
   chr   start     end strand coverage numCs numTs
1 chr1 1041001 1042000      *       18    17     1
2 chr1 2931001 2932000      *       21    21     0
3 chr1 4655001 4656000      *        3     0     3
4 chr1 4724001 4725000      *        6     4     2
5 chr1 6094001 6095000      *       18    16     2
6 chr1 9179001 9180000      *        6     6     0
--------------
sample.id: Healthy_1_100k 
assembly: h

Interesting.. I definitely don't understanf how the tiling works here.. but it looks like the deeper sequenced one has more rows? Which intuitively makes sense, but yeah I guess some other stuff has to be done downstream to merge this stuff together.

It seems to just make a somewhat straightforward dataframe?? It shouldn't be too crazy to just plug that into pandas as a csv file?

In [13]:
import pandas as pd
methobj_tiles_df = pd.read_csv("methobj_tiles.csv")
methobj_tiles_df

,chr,start,end,strand,coverage,numCs,numTs
0,chr1,1041001,1042000,*,18,17,1
1,chr1,2931001,2932000,*,21,21,0
2,chr1,4655001,4656000,*,3,0,3
3,chr1,4724001,4725000,*,6,4,2
4,chr1,6094001,6095000,*,18,16,2
...,...,...,...,...,...,...,...
326,phage_lambda,47001,48000,*,24,0,24
327,phage_lambda,48001,49000,*,12,0,12
328,plasmid_puc19c,1,1000,*,3991,3890,101
329,plasmid_puc19c,1001,2000,*,2241,2192,49


Well there you go? Pretty sure those are all the columns needed by the downstream stuff??

In [19]:
import numpy as np
chromosome_column_index = 0
start_column_index = 1
end_column_index = 2
strand_column_index = 3
coverage_column_index = 4
meth_column_index = 5
unmeth_column_index = 6
for i,tiled_counts_obj in enumerate(methobj_tiles_df):
	print(i,tiled_counts_obj)
	chr_list = [as_char(tiled_counts_obj[chromosome_column_index].rx[int(i)])[0] for i in np.arange(1, len(tiled_counts_obj[chromosome_column_index])+1)]



0 chr


NameError: name 'as_char' is not defined

Hmm.. maybe I'm actually missing something here??

In [2]:
from rpy2.robjects.packages import importr
methylKit = importr('methylKit')

In [4]:
import rpy2
methobj_lowCov = rpy2.robjects.r.readRDS("/efs/scratch/jeffrey/code/github.com/exosomedx/methylseq/tests/results/test_NSQCAM529_subsample_100k/methylqc/methobj_lowcov.rds")
methobj_lowCov


chr,[RTYPES.STRSXP]
start,[RTYPES.INTSXP]
end,[RTYPES.INTSXP]
strand,[RTYPES.STRSXP]
coverage,[RTYPES.INTSXP]
numCs,[RTYPES.INTSXP]
numTs,[RTYPES.INTSXP]


Hmm.. what am I missing??

In [5]:
methobj_tiles = methylKit.tileMethylCounts(methobj_lowCov, win_size = 1000, step_size = 1000, cov_bases = 0)
methobj_tiles

chr,[RTYPES.INTSXP]
start,[RTYPES.INTSXP]
end,[RTYPES.INTSXP]
strand,[RTYPES.INTSXP]
coverage,[RTYPES.INTSXP]
numCs,[RTYPES.INTSXP]
numTs,[RTYPES.INTSXP]


In [17]:
import rpy2.rinterface as rinterface
as_char = rinterface.baseenv['as.character']
import numpy as np

chromosome_column_index = 0

for i,tiled_counts_obj in enumerate(methobj_tiles):
	print(i,len(tiled_counts_obj))
	chr_list = [as_char(tiled_counts_obj[chromosome_column_index].rx[int(i)])[0] for i in np.arange(1, len(tiled_counts_obj[chromosome_column_index])+1)]



0 331


TypeError: object of type 'int' has no len()

1,0,3,...,101,49,11
